In [3]:
from pathlib import Path
import json
from docling.document_converter import DocumentConverter

pdf_path = Path("/home/enma/Projects/Med360_RAG_Bot/data/pdf/cardiology_all.pdf")
output_dir = Path("/home/enma/Projects/Med360_RAG_Bot/data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

stem = pdf_path.stem
master_json_path = output_dir / f"{stem}.docling.json"
markdown_path = output_dir / f"{stem}.md"

converter = DocumentConverter()
result = converter.convert(str(pdf_path))
doc = result.document

with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(doc.export_to_dict(), f, ensure_ascii=False, indent=2)

with open(markdown_path, "w", encoding="utf-8") as f:
    f.write(doc.export_to_markdown())

print(master_json_path)
print(markdown_path)

[INFO] 2026-03-10 15:12:38,827 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-10 15:12:38,828 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-10 15:12:38,853 [RapidOCR] download_file.py:60: File exists and is valid: /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-10 15:12:38,854 [RapidOCR] main.py:50: Using /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-10 15:12:39,170 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-10 15:12:39,171 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-03-10 15:12:39,175 [RapidOCR] download_file.py:60: File exists and is valid: /home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-10 15:12:39,175 [RapidOCR] main.py:50: Using /home/enma/miniconda3/envs

/home/enma/Projects/Med360_RAG_Bot/data/processed/cardiology_all.docling.json
/home/enma/Projects/Med360_RAG_Bot/data/processed/cardiology_all.md


In [4]:
#!/usr/bin/env python3
"""
02_clean_md.py

Purpose:
- Clean raw markdown from Docling
- Normalize headings, bullets, spacing
- Remove obvious extraction noise
- Preserve placeholders for images/tables for later repair
- Add source metadata block for RAG

Usage:
python 02_clean_md.py \
  --raw_md_dir /home/enma/Projects/Med360_RAG_Bot/data/extracted/raw_md \
  --output_dir /home/enma/Projects/Med360_RAG_Bot/data/processed/clean_md
"""

from __future__ import annotations

import argparse
import re
from pathlib import Path
from datetime import datetime


def now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"


def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def normalize_newlines(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n")


def collapse_spaces(text: str) -> str:
    lines = []
    for line in text.split("\n"):
        line = re.sub(r"[ \t]+", " ", line).strip()
        lines.append(line)
    return "\n".join(lines)


def remove_redundant_blank_lines(text: str) -> str:
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip() + "\n"


def normalize_bullets(text: str) -> str:
    lines = []
    for line in text.split("\n"):
        if re.match(r"^[•·▪◦]\s*", line):
            line = re.sub(r"^[•·▪◦]\s*", "- ", line)
        lines.append(line)
    return "\n".join(lines)


def normalize_headings(text: str) -> str:
    """
    Heuristic:
    - If a short line is standalone and followed by text, promote it to heading
    - Avoid changing lines that already start with markdown heading markers
    """
    lines = text.split("\n")
    out = []

    for i, line in enumerate(lines):
        stripped = line.strip()

        if not stripped:
            out.append("")
            continue

        if stripped.startswith("#"):
            out.append(stripped)
            continue

        prev_blank = i == 0 or not lines[i - 1].strip()
        next_exists = i + 1 < len(lines)
        next_nonempty = next_exists and bool(lines[i + 1].strip())

        looks_like_heading = (
            prev_blank
            and next_nonempty
            and len(stripped) <= 80
            and not stripped.endswith(".")
            and stripped.lower() == stripped.lower()
            and len(stripped.split()) <= 10
            and not re.match(r"^[-*]\s+", stripped)
            and "<!-- image -->" not in stripped
        )

        if looks_like_heading:
            out.append(f"## {stripped}")
        else:
            out.append(stripped)

    return "\n".join(out)


def fix_hyphenated_linebreaks(text: str) -> str:
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    return text


def join_wrapped_lines(text: str) -> str:
    """
    Joins lines that are likely just wrapped paragraph text,
    but preserves headings, bullets, tables, and image placeholders.
    """
    lines = text.split("\n")
    merged = []
    buffer = []

    def flush_buffer():
        nonlocal buffer, merged
        if buffer:
            merged.append(" ".join(buffer).strip())
            buffer = []

    for line in lines:
        stripped = line.strip()

        special = (
            not stripped
            or stripped.startswith("#")
            or stripped.startswith("- ")
            or stripped.startswith("|")
            or stripped == "<!-- image -->"
            or re.match(r"^\d+\.\s+", stripped)
        )

        if special:
            flush_buffer()
            merged.append(stripped)
        else:
            if buffer and re.search(r"[.:;!?)]$", buffer[-1]):
                flush_buffer()
            buffer.append(stripped)

    flush_buffer()
    return "\n".join(merged)


def remove_duplicate_lines(text: str) -> str:
    lines = text.split("\n")
    out = []
    prev = None
    for line in lines:
        if line == prev and line.strip():
            continue
        out.append(line)
        prev = line
    return "\n".join(out)


def normalize_image_placeholders(text: str) -> str:
    count = 0
    lines = []
    for line in text.split("\n"):
        if "<!-- image -->" in line:
            count += 1
            lines.append(f"## Figure Placeholder {count}\n[IMAGE_PLACEHOLDER_{count}]")
        else:
            lines.append(line)
    return "\n".join(lines)


def add_metadata_block(text: str, source_file: str, doc_type: str = "medical_pdf") -> str:
    title = None
    for line in text.split("\n"):
        stripped = line.strip()
        if stripped.startswith("# "):
            title = stripped[2:].strip()
            break

    if title is None:
        title = Path(source_file).stem.replace("_", " ").title()
        text = f"# {title}\n\n{text}"

    meta = (
        f"## Source Metadata\n"
        f"- source_file: {Path(source_file).name}\n"
        f"- source_type: {doc_type}\n"
        f"- cleaned_on: {now_iso()}\n\n"
    )

    if "## Source Metadata" not in text:
        parts = text.split("\n", 2)
        if len(parts) >= 2 and parts[0].startswith("# "):
            text = parts[0] + "\n\n" + meta + ("\n".join(parts[1:]).lstrip())
        else:
            text = meta + text

    return text


def clean_markdown(raw_text: str, source_file: str) -> str:
    text = raw_text
    text = normalize_newlines(text)
    text = fix_hyphenated_linebreaks(text)
    text = collapse_spaces(text)
    text = normalize_bullets(text)
    text = join_wrapped_lines(text)
    text = normalize_headings(text)
    text = normalize_image_placeholders(text)
    text = remove_duplicate_lines(text)
    text = remove_redundant_blank_lines(text)
    text = add_metadata_block(text, source_file=source_file)
    text = remove_redundant_blank_lines(text)
    return text


def process_file(md_path: Path, output_dir: Path) -> None:
    raw_text = read_text(md_path)
    source_file = md_path.name.replace(".raw.md", ".pdf")
    cleaned = clean_markdown(raw_text, source_file=source_file)

    out_path = output_dir / md_path.name.replace(".raw.md", ".clean.md")
    write_text(out_path, cleaned)
    print(f"[OK] {md_path.name}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw_md_dir", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    args = parser.parse_args()

    raw_md_dir = Path(args.raw_md_dir)
    output_dir = Path(args.output_dir)

    files = sorted(raw_md_dir.glob("*.md"))
    if not files:
        print("No markdown files found.")
        return

    for md_path in files:
        process_file(md_path, output_dir)


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --raw_md_dir RAW_MD_DIR --output_dir
                             OUTPUT_DIR
ipykernel_launcher.py: error: the following arguments are required: --raw_md_dir, --output_dir


SystemExit: 2

/home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import os
import re
import json
import time
import hashlib
import mimetypes
from pathlib import Path
from urllib.parse import urljoin, urlparse, urldefrag

import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader

# Docling
from docling.document_converter import DocumentConverter


# =========================================================
# CONFIG
# =========================================================
START_URLS = [
    "https://www.icmr.gov.in/guidelines",
    "https://www.icmr.gov.in/reports",
    "https://www.icmr.gov.in/downloadable-books",
    "https://www.icmr.gov.in/annual-reports",
]

OUTPUT_ROOT = Path("data_corpus")
DOWNLOAD_DIR = OUTPUT_ROOT / "downloads"
PARSED_DIR = OUTPUT_ROOT / "parsed"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
LOG_DIR = OUTPUT_ROOT / "logs"

MANIFEST_PATH = MANIFEST_DIR / "corpus_manifest.jsonl"
LOG_PATH = LOG_DIR / "scrape_log.txt"

REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_REQUESTS = 0.8
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0 Safari/537.36"
)

MAX_PAGES_PER_SITE = 100
FOLLOW_INTERNAL_LINKS = True
PDF_EXTENSIONS = {".pdf"}

# If True, do not redownload files already present
SKIP_EXISTING_DOWNLOADS = True


# =========================================================
# SETUP
# =========================================================
for d in [DOWNLOAD_DIR, PARSED_DIR, MANIFEST_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

doc_converter = DocumentConverter()


# =========================================================
# HELPERS
# =========================================================
def log(msg: str):
    print(msg)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_filename(name: str, max_len: int = 180) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^\w\s\-.]", "_", name)
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"_+", "_", name).strip("._")
    if not name:
        name = "document"
    return name[:max_len]


def normalize_url(url: str) -> str:
    url, _ = urldefrag(url)
    return url.strip()


def is_pdf_url(url: str) -> bool:
    parsed = urlparse(url)
    path = parsed.path.lower()
    return any(path.endswith(ext) for ext in PDF_EXTENSIONS)


def guess_domain_label(url: str) -> str:
    netloc = urlparse(url).netloc.lower()
    if "icmr" in netloc:
        return "ICMR"
    if "mohfw" in netloc:
        return "MoHFW"
    if "ncdc" in netloc:
        return "NCDC"
    if "who.int" in netloc:
        return "WHO"
    return netloc


def response_looks_like_pdf(resp: requests.Response) -> bool:
    ctype = resp.headers.get("Content-Type", "").lower()
    return "application/pdf" in ctype


def extract_links_from_html(base_url: str, html: str) -> tuple[list[str], list[str]]:
    soup = BeautifulSoup(html, "html.parser")
    pdf_links = []
    page_links = []

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href:
            continue

        full_url = normalize_url(urljoin(base_url, href))
        parsed = urlparse(full_url)

        if parsed.scheme not in {"http", "https"}:
            continue

        if is_pdf_url(full_url):
            pdf_links.append(full_url)
        else:
            if FOLLOW_INTERNAL_LINKS:
                base_netloc = urlparse(base_url).netloc
                if parsed.netloc == base_netloc:
                    page_links.append(full_url)

    return sorted(set(pdf_links)), sorted(set(page_links))


def fetch_url(url: str) -> requests.Response | None:
    try:
        resp = session.get(url, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        return resp
    except Exception as e:
        log(f"[ERROR] fetch failed: {url} | {e}")
        return None


def crawl_for_pdfs(start_urls: list[str], max_pages_per_site: int = 100) -> list[dict]:
    """
    Returns list of dicts:
    {
      "pdf_url": ...,
      "source_page": ...,
      "source_domain": ...
    }
    """
    all_pdf_records = []
    seen_pages = set()
    seen_pdfs = set()

    for start_url in start_urls:
        log(f"[CRAWL] Start: {start_url}")
        queue = [start_url]
        pages_seen_for_site = 0

        while queue and pages_seen_for_site < max_pages_per_site:
            current = queue.pop(0)
            if current in seen_pages:
                continue

            seen_pages.add(current)
            pages_seen_for_site += 1

            resp = fetch_url(current)
            if resp is None:
                continue

            # Skip if page itself is a PDF
            if response_looks_like_pdf(resp) or is_pdf_url(current):
                if current not in seen_pdfs:
                    seen_pdfs.add(current)
                    all_pdf_records.append(
                        {
                            "pdf_url": current,
                            "source_page": current,
                            "source_domain": guess_domain_label(current),
                        }
                    )
                continue

            pdf_links, page_links = extract_links_from_html(current, resp.text)

            for pdf_url in pdf_links:
                if pdf_url not in seen_pdfs:
                    seen_pdfs.add(pdf_url)
                    all_pdf_records.append(
                        {
                            "pdf_url": pdf_url,
                            "source_page": current,
                            "source_domain": guess_domain_label(pdf_url),
                        }
                    )

            for link in page_links:
                if link not in seen_pages:
                    queue.append(link)

        log(f"[CRAWL] Done: {start_url} | pages crawled={pages_seen_for_site}")

    return all_pdf_records


def download_pdf(pdf_url: str, source_domain: str) -> Path | None:
    try:
        parsed = urlparse(pdf_url)
        original_name = Path(parsed.path).name or "document.pdf"
        original_name = original_name if original_name.lower().endswith(".pdf") else original_name + ".pdf"

        stem = safe_filename(Path(original_name).stem)
        filename = f"{safe_filename(source_domain)}__{stem}.pdf"
        pdf_path = DOWNLOAD_DIR / filename

        if pdf_path.exists() and SKIP_EXISTING_DOWNLOADS:
            log(f"[SKIP] already downloaded: {pdf_path.name}")
            return pdf_path

        resp = session.get(pdf_url, timeout=REQUEST_TIMEOUT, stream=True)
        resp.raise_for_status()

        content_type = resp.headers.get("Content-Type", "").lower()
        if "pdf" not in content_type and not pdf_url.lower().endswith(".pdf"):
            log(f"[WARN] not obviously a pdf: {pdf_url} | content-type={content_type}")

        with open(pdf_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        time.sleep(SLEEP_BETWEEN_REQUESTS)
        log(f"[DOWNLOADED] {pdf_url} -> {pdf_path}")
        return pdf_path

    except Exception as e:
        log(f"[ERROR] download failed: {pdf_url} | {e}")
        return None


def read_pdf_basic_metadata(pdf_path: Path) -> dict:
    info = {
        "page_count": None,
        "pdf_metadata": {},
        "plain_text_chars": 0,
        "plain_text_preview": "",
        "is_encrypted": None,
    }

    try:
        reader = PdfReader(str(pdf_path))
        info["page_count"] = len(reader.pages)
        info["is_encrypted"] = bool(reader.is_encrypted)

        meta = reader.metadata or {}
        pdf_meta = {}
        for k, v in meta.items():
            try:
                pdf_meta[str(k)] = str(v)
            except Exception:
                pdf_meta[str(k)] = None
        info["pdf_metadata"] = pdf_meta

        text_parts = []
        for page in reader.pages[:10]:  # preview only for speed
            try:
                txt = page.extract_text() or ""
                if txt.strip():
                    text_parts.append(txt)
            except Exception:
                pass

        preview_text = "\n".join(text_parts).strip()
        info["plain_text_chars"] = len(preview_text)
        info["plain_text_preview"] = preview_text[:3000]

    except Exception as e:
        info["pdf_read_error"] = str(e)

    return info


def fallback_extract_full_text(pdf_path: Path) -> str:
    parts = []
    try:
        reader = PdfReader(str(pdf_path))
        for page in reader.pages:
            try:
                txt = page.extract_text() or ""
                if txt:
                    parts.append(txt)
            except Exception:
                continue
    except Exception:
        pass
    return "\n\n".join(parts).strip()


def try_docling_parse(pdf_path: Path, out_dir: Path) -> dict:
    """
    Returns:
    {
      "docling_ok": bool,
      "markdown_path": str | None,
      "json_path": str | None,
      "docling_error": str | None
    }
    """
    result = {
        "docling_ok": False,
        "markdown_path": None,
        "json_path": None,
        "docling_error": None,
    }

    try:
        conv_result = doc_converter.convert(str(pdf_path))
        doc = conv_result.document

        md_path = out_dir / "parsed.md"
        json_path = out_dir / "parsed.json"

        # These methods are commonly available in Docling workflows
        md_text = doc.export_to_markdown()
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        doc_dict = doc.export_to_dict()
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(doc_dict, f, ensure_ascii=False, indent=2)

        result["docling_ok"] = True
        result["markdown_path"] = str(md_path)
        result["json_path"] = str(json_path)

    except Exception as e:
        result["docling_error"] = str(e)

    return result


def build_doc_id(source_domain: str, pdf_path: Path, sha256: str) -> str:
    stem = safe_filename(pdf_path.stem)
    return f"{safe_filename(source_domain)}__{stem}__{sha256[:12]}"


def infer_document_type(title: str, url: str) -> str:
    txt = f"{title} {url}".lower()
    if "guideline" in txt:
        return "guideline"
    if "report" in txt:
        return "report"
    if "book" in txt:
        return "book"
    if "manual" in txt:
        return "manual"
    if "circular" in txt:
        return "circular"
    if "protocol" in txt:
        return "protocol"
    return "unknown"


def infer_specialty(text: str) -> list[str]:
    t = text.lower()
    tags = []

    mapping = {
        "cardiology": ["cardiac", "heart", "cardio", "hypertension", "ecg"],
        "endocrinology": ["diabetes", "thyroid", "hba1c", "insulin"],
        "nephrology": ["kidney", "renal", "creatinine", "urea"],
        "hepatology": ["liver", "bilirubin", "sgot", "sgpt", "alp"],
        "hematology": ["hemoglobin", "cbc", "anemia", "platelet", "wbc"],
        "infectious_disease": ["infection", "tb", "tuberculosis", "malaria", "viral", "bacterial"],
        "pulmonology": ["lung", "respiratory", "asthma", "copd"],
        "oncology": ["cancer", "tumor", "oncology"],
        "gynecology": ["pregnancy", "maternal", "obstetric", "gyne", "antenatal"],
        "pediatrics": ["child", "children", "pediatric", "newborn", "neonate"],
        "primary_care": ["screening", "follow up", "primary care", "outpatient", "opd"],
    }

    for tag, keywords in mapping.items():
        if any(k in t for k in keywords):
            tags.append(tag)

    return sorted(set(tags))


def write_json(path: Path, obj: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def append_manifest_record(record: dict):
    with open(MANIFEST_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# =========================================================
# MAIN PIPELINE
# =========================================================
def process_pdf_record(pdf_record: dict):
    pdf_url = pdf_record["pdf_url"]
    source_page = pdf_record["source_page"]
    source_domain = pdf_record["source_domain"]

    pdf_path = download_pdf(pdf_url, source_domain)
    if pdf_path is None:
        return

    file_sha256 = sha256_file(pdf_path)
    doc_id = build_doc_id(source_domain, pdf_path, file_sha256)
    out_dir = PARSED_DIR / doc_id
    out_dir.mkdir(parents=True, exist_ok=True)

    # Copy source pdf into document folder as canonical local copy
    canonical_pdf_path = out_dir / "source.pdf"
    if not canonical_pdf_path.exists():
        canonical_pdf_path.write_bytes(pdf_path.read_bytes())

    basic = read_pdf_basic_metadata(canonical_pdf_path)
    docling_info = try_docling_parse(canonical_pdf_path, out_dir)

    fallback_txt = fallback_extract_full_text(canonical_pdf_path)
    fallback_txt_path = out_dir / "fallback.txt"
    with open(fallback_txt_path, "w", encoding="utf-8") as f:
        f.write(fallback_txt)

    # Decide best available text for metadata inference
    best_text = ""
    md_path = out_dir / "parsed.md"
    if md_path.exists():
        best_text = md_path.read_text(encoding="utf-8", errors="ignore")
    elif fallback_txt:
        best_text = fallback_txt
    else:
        best_text = basic.get("plain_text_preview", "")

    title = (
        basic.get("pdf_metadata", {}).get("/Title")
        or basic.get("pdf_metadata", {}).get("Title")
        or pdf_path.stem
    )
    document_type = infer_document_type(title, pdf_url)
    specialty_tags = infer_specialty((title or "") + "\n" + best_text[:8000])

    metadata = {
        "doc_id": doc_id,
        "title": title,
        "source_name": source_domain,
        "source_url": pdf_url,
        "discovered_from": source_page,
        "document_type": document_type,
        "specialty_tags": specialty_tags,
        "country": "India" if source_domain in {"ICMR", "MoHFW", "NCDC"} else None,
        "language": "unknown",

        "local_paths": {
            "source_pdf": str(canonical_pdf_path),
            "parsed_markdown": docling_info["markdown_path"],
            "parsed_json": docling_info["json_path"],
            "fallback_text": str(fallback_txt_path),
        },

        "file_info": {
            "filename": canonical_pdf_path.name,
            "size_bytes": canonical_pdf_path.stat().st_size,
            "mime_type": mimetypes.guess_type(str(canonical_pdf_path))[0],
            "sha256": file_sha256,
        },

        "pdf_info": {
            "page_count": basic.get("page_count"),
            "is_encrypted": basic.get("is_encrypted"),
            "embedded_metadata": basic.get("pdf_metadata", {}),
        },

        "parse_info": {
            "docling_ok": docling_info["docling_ok"],
            "docling_error": docling_info["docling_error"],
            "fallback_text_chars": len(fallback_txt),
            "preview_text_chars": basic.get("plain_text_chars"),
        },

        "quality_flags": {
            "has_markdown": bool(docling_info["markdown_path"]),
            "has_json": bool(docling_info["json_path"]),
            "has_fallback_text": bool(fallback_txt.strip()),
            "text_extraction_usable": len(best_text.strip()) > 500,
            "citation_ready": basic.get("page_count") is not None,
        },

        "status": {
            "doc_status": "raw",
            "serve_level": "low",
            "cleaning_status": "not_cleaned",
            "chunking_status": "not_chunked",
            "review_status": "not_reviewed",
        },

        "created_by_pipeline": {
            "crawler": "requests+bs4",
            "parser_primary": "docling",
            "parser_fallback": "pypdf",
            "version": "v1",
        },
    }

    metadata_path = out_dir / "metadata.json"
    write_json(metadata_path, metadata)

    manifest_record = {
        "doc_id": doc_id,
        "title": title,
        "source_name": source_domain,
        "source_url": pdf_url,
        "discovered_from": source_page,
        "document_type": document_type,
        "page_count": basic.get("page_count"),
        "sha256": file_sha256,
        "docling_ok": docling_info["docling_ok"],
        "text_extraction_usable": metadata["quality_flags"]["text_extraction_usable"],
        "citation_ready": metadata["quality_flags"]["citation_ready"],
        "serve_level": metadata["status"]["serve_level"],
        "metadata_path": str(metadata_path),
        "parsed_markdown": docling_info["markdown_path"],
        "parsed_json": docling_info["json_path"],
        "fallback_text": str(fallback_txt_path),
    }
    append_manifest_record(manifest_record)

    log(f"[PROCESSED] {doc_id}")


def main():
    pdf_records = crawl_for_pdfs(START_URLS, max_pages_per_site=MAX_PAGES_PER_SITE)
    log(f"[SUMMARY] total discovered pdf links = {len(pdf_records)}")

    for i, pdf_record in enumerate(pdf_records, start=1):
        log(f"[{i}/{len(pdf_records)}] Processing: {pdf_record['pdf_url']}")
        process_pdf_record(pdf_record)

    log("[DONE] Corpus scrape + parse finished.")


if __name__ == "__main__":
    main()

/home/enma/miniconda3/envs/medrag_all/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


[CRAWL] Start: https://www.icmr.gov.in/guidelines
